# Pasos 12 y 13: variaciones y evaluación final

Este cuaderno continúa `prompt_engineering_lab.ipynb`. Usa los prompts v3 guardados en `resultados/` y no repite las series anteriores. Ejecutar las celdas en orden, una a una, con Mayús + Intro.

1. **Preparación** (sin llamadas a la API): carga la configuración y las funciones de evaluación.
2. **Paso 12**: variaciones de las tres tareas (unas 35 llamadas en total, 2 en paralelo).
3. **Paso 13**: tabla final v1, v2 y v3 calculada desde los archivos guardados (sin llamadas a la API).

## Preparación (sin llamadas a la API)

In [1]:
import os, re, json, time, datetime
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-3.1-flash-lite-preview"

def call_llm(prompt, temperature=0.7, max_tokens=500, reintentos=3):
    for intento in range(reintentos + 1):
        try:
            respuesta = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return respuesta.choices[0].message.content
        except Exception as e:
            texto = str(e)
            temporal = "503" in texto or "429" in texto
            if temporal and intento < reintentos:
                time.sleep(2 * (intento + 1))
                continue
            return f"Error: {e}"

def run_n_par(prompt, n, workers=5):
    with ThreadPoolExecutor(max_workers=workers) as pool:
        return list(pool.map(lambda _: call_llm(prompt), range(n)))

os.makedirs("resultados", exist_ok=True)

def guardar_serie(nombre, prompt, respuestas, temperature=0.7):
    datos = {
        "nombre": nombre,
        "modelo": MODEL,
        "temperature": temperature,
        "fecha": datetime.datetime.now().isoformat(timespec="seconds"),
        "prompt": prompt,
        "respuestas": respuestas,
    }
    with open(f"resultados/{nombre}.json", "w", encoding="utf-8") as f:
        json.dump(datos, f, ensure_ascii=False, indent=2)

def es_valida_sentimiento(r):
    return r.strip().strip(".").lower() in {"positive", "negative", "neutral"}

def evaluar_producto(r):
    t = r.strip()
    return {
        "palabras_40_60": 40 <= len(t.split()) <= 60,
        "un_parrafo": "\n" not in t,
        "sin_vinetas": not re.search(r"^\s*([*\-•#]|\d+\.)\s", t, re.M),
        "sin_numeros_extra": not re.search(r"\d", t.replace("29.99", "")),
    }

def es_valida_producto(r):
    return all(evaluar_producto(r).values())

CLAVES = {"order_number", "order_date", "delivery", "packaging"}

def evaluar_extraccion(r):
    try:
        d = json.loads(r.strip())
    except Exception:
        return {"json_valido": False, "cuatro_claves": False, "pedido_ok": False, "fecha_ok": False}
    ok_dict = isinstance(d, dict)
    return {
        "json_valido": True,
        "cuatro_claves": ok_dict and set(d.keys()) == CLAVES,
        "pedido_ok": ok_dict and "12345" in str(d.get("order_number", "")),
        "fecha_ok": ok_dict and "March 15" in str(d.get("order_date", "")),
    }

def es_valida_extraccion(r):
    return all(evaluar_extraccion(r).values())

print("Modelo:", MODEL)
print("Clave encontrada:", bool(os.getenv("GEMINI_API_KEY")))
print("Preparación lista: funciones cargadas, ninguna llamada al modelo.")

Modelo: gemini-3.1-flash-lite-preview
Clave encontrada: True
Preparación lista: funciones cargadas, ninguna llamada al modelo.


## Paso 12: variaciones de las tareas

Propuesta de la IA, a revisar por Jaime. Se prueban los prompts v3 con otras entradas para ver si aguantan. Para cuidar la cuota gratuita de la API, cada variación se ejecuta 5 veces (no 15) con 2 llamadas en paralelo.

- **Sentimiento:** un mensaje negativo, uno neutro (una pregunta) y uno mixto, sin una única respuesta correcta.
- **Producto:** otros dos productos con otro precio (teclado y botella).
- **Extracción:** otra opinión con los mismos campos y una opinión sin número de pedido ni fecha, para ver si el modelo inventa datos.

In [2]:
def cargar_prompt(nombre):
    with open(f"resultados/{nombre}.json", encoding="utf-8") as f:
        return json.load(f)["prompt"]

def tanda(nombre, prompt, n=5):
    resp = run_n_par(prompt, n, workers=2)
    guardar_serie(nombre, prompt, resp)
    reales = [r for r in resp if not r.startswith("Error")]
    print(f"{nombre}: {len(resp)} ejecuciones, {len(resp) - len(reales)} errores, {len(reales)} reales")
    return reales

# --- Sentimiento ---
base_s = cargar_prompt("sentimiento_v3").rsplit('Message: "', 1)[0]
mensajes_s = {
    "negativo": "The mouse stopped working after two days and support never replied.",
    "neutro": "I would like to know if this model comes in black.",
    "mixto": "The screen is beautiful but the battery barely lasts a day.",
}
resultados_s = {}
for etiqueta, mensaje in mensajes_s.items():
    p = base_s + f'Message: "{mensaje}"\nSentiment:'
    resultados_s[etiqueta] = tanda(f"variacion_sentimiento_{etiqueta}", p)
print()
for etiqueta, reales in resultados_s.items():
    cuenta = {}
    for r in reales:
        clave = r.strip().strip(".")
        cuenta[clave] = cuenta.get(clave, 0) + 1
    print(f"Sentimiento {etiqueta}:", cuenta)

variacion_sentimiento_negativo: 5 ejecuciones, 0 errores, 5 reales
variacion_sentimiento_neutro: 5 ejecuciones, 0 errores, 5 reales
variacion_sentimiento_mixto: 5 ejecuciones, 0 errores, 5 reales

Sentimiento negativo: {'Negative': 5}
Sentimiento neutro: {'Neutral': 5}
Sentimiento mixto: {'Negative': 5}


In [3]:
# --- Producto ---
pv3 = cargar_prompt("producto_v3")
cola = "Product: wireless mouse, $29.99\nDescription:"
assert cola in pv3
base_p = pv3.rsplit(cola, 1)[0]

def evaluar_producto_gen(r, precio):
    t = r.strip()
    return {
        "palabras_40_60": 40 <= len(t.split()) <= 60,
        "un_parrafo": "\n" not in t,
        "sin_vinetas": not re.search(r"^\s*([*\-•#]|\d+\.)\s", t, re.M),
        "sin_numeros_extra": not re.search(r"\d", t.replace(precio, "")),
    }

productos = {"teclado": ("keyboard", "$59.90", "59.90"), "botella": ("water bottle", "$18.50", "18.50")}
for etiqueta, (nombre_prod, precio, precio_num) in productos.items():
    p = base_p + f"Product: {nombre_prod}, {precio}\nDescription:"
    reales = tanda(f"variacion_producto_{etiqueta}", p)
    validas = [r for r in reales if all(evaluar_producto_gen(r, precio_num).values())]
    print(f"  Producto {etiqueta}: válidas {len(validas)} de {len(reales)}")
    for criterio in ["palabras_40_60", "un_parrafo", "sin_vinetas", "sin_numeros_extra"]:
        print(f"    {criterio}:", sum(evaluar_producto_gen(r, precio_num)[criterio] for r in reales), "de", len(reales))
    print("   ", reales[0].replace("\n", " ")[:200] if reales else "sin respuestas")

variacion_producto_teclado: 5 ejecuciones, 3 errores, 2 reales
  Producto teclado: válidas 2 de 2
    palabras_40_60: 2 de 2
    un_parrafo: 2 de 2
    sin_vinetas: 2 de 2
    sin_numeros_extra: 2 de 2
    This keyboard is a reliable choice for your workspace, offering a practical solution for all your typing needs. Designed with simplicity in mind, it provides a straightforward experience for daily use
variacion_producto_botella: 5 ejecuciones, 0 errores, 5 reales
  Producto botella: válidas 5 de 5
    palabras_40_60: 5 de 5
    un_parrafo: 5 de 5
    sin_vinetas: 5 de 5
    sin_numeros_extra: 5 de 5
    This water bottle is an accessible choice for anyone needing a reliable way to stay hydrated throughout the day. Designed for simplicity and convenience, it serves as a practical companion for your da


In [4]:
# --- Extracción ---
ev3 = cargar_prompt("extraccion_v3")
base_e = ev3.split("Customer feedback:")[0]

def json_final(r):
    if "FINAL JSON:" not in r:
        return None
    parte = r.split("FINAL JSON:")[-1].strip()
    if parte.startswith("```"):
        parte = parte.strip("`").strip()
        if parte.lower().startswith("json"):
            parte = parte[4:].strip()
    return parte

# Variación A: otra opinión con los mismos campos
texto_a = "I ordered item #98765 on June 3rd. The product works great but the delivery took two weeks."
reales_a = tanda("variacion_extraccion_otra_opinion", base_e + f'Customer feedback: "{texto_a}"\n')
ok_a = 0
for r in reales_a:
    parte = json_final(r)
    try:
        d = json.loads(parte) if parte else None
    except Exception:
        d = None
    if isinstance(d, dict) and set(d.keys()) == CLAVES and "98765" in str(d.get("order_number", "")) and "June 3" in str(d.get("order_date", "")):
        ok_a += 1
print("  Opinión con datos completos: válidas", ok_a, "de", len(reales_a))

# Variación B: opinión sin número de pedido ni fecha (¿inventa datos?)
texto_b = "The mug arrived broken and I would like a refund."
reales_b = tanda("variacion_extraccion_sin_datos", base_e + f'Customer feedback: "{texto_b}"\n')
con_digitos = 0
for r in reales_b:
    parte = json_final(r)
    try:
        d = json.loads(parte) if parte else None
    except Exception:
        d = None
    if isinstance(d, dict) and re.search(r"\d", str(d.get("order_number", ""))):
        con_digitos += 1
print("  Opinión sin datos: respuestas con dígitos en order_number (posible invención):", con_digitos, "de", len(reales_b))
for r in reales_b[:2]:
    print("   ", (json_final(r) or r[-150:]).replace("\n", " ")[:200])

variacion_extraccion_otra_opinion: 5 ejecuciones, 0 errores, 5 reales
  Opinión con datos completos: válidas 5 de 5
variacion_extraccion_sin_datos: 5 ejecuciones, 0 errores, 5 reales
  Opinión sin datos: respuestas con dígitos en order_number (posible invención): 0 de 5
    {   "order_number": null,   "order_date": null,   "delivery": "The mug arrived broken",   "packaging": null }
    {   "order_number": null,   "order_date": null,   "delivery": "The mug arrived broken",   "packaging": null }


## Paso 13: evaluación final y comparación

Se recalcula, a partir de los archivos guardados en `resultados/`, la consistencia de las tres versiones con los mismos criterios de respuesta válida. No hace llamadas a la API.

In [5]:
def json_final_cmp(r):
    if "FINAL JSON:" not in r:
        return r
    parte = r.split("FINAL JSON:")[-1].strip()
    if parte.startswith("```"):
        parte = parte.strip("`").strip()
        if parte.lower().startswith("json"):
            parte = parte[4:].strip()
    return parte

criterios = {
    "sentimiento": es_valida_sentimiento,
    "producto": es_valida_producto,
    "extraccion": lambda r: es_valida_extraccion(json_final_cmp(r)),
}

print(f"{'Tarea':<12}{'Versión':<9}{'Errores':<9}{'Reales':<8}{'Válidas':<9}{'Consistencia'}")
for tarea, criterio in criterios.items():
    for version in ["v1", "v2", "v3"]:
        with open(f"resultados/{tarea}_{version}.json", encoding="utf-8") as f:
            resp = json.load(f)["respuestas"]
        reales = [r for r in resp if not r.startswith("Error")]
        validas = [r for r in reales if criterio(r)]
        pct = round(100 * len(validas) / len(reales), 1) if reales else 0
        print(f"{tarea:<12}{version:<9}{len(resp) - len(reales):<9}{len(reales):<8}{len(validas):<9}{pct} %")

print()
print("Nota: la extracción v1 no pedía formato JSON, por eso aparece en 0 %, aunque sus 4 datos eran correctos en 15 de 15.")

Tarea       Versión  Errores  Reales  Válidas  Consistencia
sentimiento v1       3        12      0        0.0 %
sentimiento v2       0        15      15       100.0 %
sentimiento v3       0        15      15       100.0 %
producto    v1       3        12      0        0.0 %
producto    v2       0        15      15       100.0 %
producto    v3       0        15      15       100.0 %
extraccion  v1       0        15      0        0.0 %
extraccion  v2       0        15      15       100.0 %
extraccion  v3       4        11      11       100.0 %

Nota: la extracción v1 no pedía formato JSON, por eso aparece en 0 %, aunque sus 4 datos eran correctos en 15 de 15.
